# Part B — Customer Segments and Lapse Prediction
This notebook is the **main runner**. All processing lives in `part_b.py`. Run cells in order after Part A produces the two clean CSVs.

In [4]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent
if not (ROOT / "generate_data.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)
print("Root files:", sorted(p.name for p in ROOT.iterdir() if p.is_file()))

Project root: d:\Sowbarnika\Instamat_project
Root files: ['README.md', 'generate_data.py', 'mock_llm.py', 'requirements.txt', 'validate.py']


In [5]:
from src.part_b import (load_data, build_rfm, cluster_segments, build_lapse_cohort,
                    train_evaluate, balance_experiment, interpret_persist_retain)

## 1. Load clean Part A data
Read clean orders and customers, validate required columns, parse timestamps and standardize order status.

In [6]:
orders, customers = load_data()
print(orders.shape, customers.shape)

(12309, 10) (1200, 5)


## 2. Build RFM (Task 1)
At 2025-10-01, calculate days since last completed order, distinct completed orders and total completed spending.

In [7]:
rfm = build_rfm(orders)
display(rfm.head())

RFM customers: 1175 median/min/max recency: 19.0 0 180
Frequency mean/max: 9.45531914893617 33 Monetary mean/max: 2552.0585531914894 14766.550000000001


,customer_id,recency,frequency,monetary
0,C0001,14,4,512.85
1,C0002,28,7,4644.90
2,C0003,7,22,3644.85
3,C0004,0,5,369.00
4,C0005,147,2,324.00


## 3. Compare K-means and name segments (Tasks 2–3)
Standardize RFM; inspect K=2–8 inertia and silhouette; use the interpretable K=4 solution. Premium is always the highest-spending mean cluster. Show the two-component PCA explained variance and save `customer_segments.csv`. Cluster IDs are arbitrary.

In [8]:
segments, k_scores, segment_profile, pca = cluster_segments(rfm, k=4)
display(k_scores, segment_profile, segments.head())

 K     inertia  silhouette
 2 1820.760553    0.464972
 3 1268.234587    0.445161
 4  743.071108    0.525544
 5  617.366636    0.480495
 6  503.170622    0.481872
 7  408.460454    0.416337
 8  366.503487    0.412329
Interpretation: K=4 is the reference choice: strong silhouette and an actionable elbow. Inertia always falls with more clusters because extra centroids cannot worsen the optimum; it does not penalize complexity or measure usefulness.

Segment profile (cluster IDs are arbitrary labels, not ranks):
             size     recency  frequency     monetary    segment_name                 recommended_action
segment_id                                                                                             
0            436   28.862385   3.772936  1057.618234      Occasional          Frequency-building offers
1            138   24.115942   6.768116  7129.695652         Premium  VIP care and personalized rewards
2            419    8.766110  19.403341  3410.020525  Frequent Loyal 

,K,inertia,silhouette
0,2,1820.760553,0.464972
1,3,1268.234587,0.445161
2,4,743.071108,0.525544
3,5,617.366636,0.480495
4,6,503.170622,0.481872
5,7,408.460454,0.416337
6,8,366.503487,0.412329


,size,recency,frequency,monetary,segment_name,recommended_action
segment_id,,,,,,
0,436,28.862385,3.772936,1057.618234,Occasional,Frequency-building offers
1,138,24.115942,6.768116,7129.695652,Premium,VIP care and personalized rewards
2,419,8.766110,19.403341,3410.020525,Frequent Loyal,Loyalty perks and cross-sell
3,182,109.043956,2.203297,685.992582,At Risk,Win-back campaign


,customer_id,recency,frequency,monetary,segment_id,segment_name,is_premium
0,C0001,14,4,512.85,0,Occasional,False
1,C0002,28,7,4644.90,1,Premium,True
2,C0003,7,22,3644.85,2,Frequent Loyal,False
3,C0004,0,5,369.00,0,Occasional,False
4,C0005,147,2,324.00,3,At Risk,False


## 4. Create leakage-free lapse cohort (Task 4)
Features use orders strictly before 2025-09-01. September completed orders determine only the target label; segment IDs and names are excluded.

In [9]:
cohort, features = build_lapse_cohort(orders, customers)
display(cohort.head())
assert not cohort[features].isna().any().any()

Cohort: 1156 lapsed: 443 (38.32%)


,customer_id,frequency,monetary,avg_discount_pct,avg_delivery_minutes,recency,cancellation_rate,favourite_category,city,age,gender,tenure_days,lapsed
0,C0001,2,378.00,10.000000,28.000000,33,0.000000,Grocery,Chennai,35.0,male,597,0
1,C0002,6,3946.90,6.666667,21.916667,22,0.000000,Electronics,Delhi,31.0,male,562,0
2,C0003,20,3264.05,12.750000,24.200000,1,0.000000,Grocery,Bengaluru,18.0,female,195,0
3,C0004,3,272.25,11.666667,35.666667,42,0.400000,Snacks,Bengaluru,24.0,female,437,0
4,C0005,2,324.00,25.000000,26.000000,117,0.333333,Dairy,Bengaluru,25.0,female,335,1


## 5. Preprocess, tune and evaluate (Tasks 5–6)
Stratified 80/20 split; train-only scaler and one-hot encoder inside pipelines. Report baseline, scaler training mean vs cohort mean, both prescribed 5-fold grids, and held-out confusion matrices and positive-class metrics. Compare false negatives (missed lapsers) against false positives (unnecessary offers).

In [10]:
run = train_evaluate(cohort, features)

Train/test: 924 232 Class counts train: {0: 570, 1: 354}
Baseline accuracy=0.616, lapsed precision=0.000, recall=0.000, F1=0.000
Logistic best: {'model__C': 100} CV F1: 0.650552962900789
Forest best: {'model__max_depth': 8, 'model__min_samples_leaf': 10} CV F1: 0.6810534884225208
Logistic: accuracy=0.728 precision=0.644 recall=0.652 F1=0.648
Confusion matrix [0,1]:
[[111  32]
 [ 31  58]]
Forest: accuracy=0.737 precision=0.659 recall=0.652 F1=0.655
Confusion matrix [0,1]:
[[113  30]
 [ 31  58]]
Training frequency scaler mean: 8.114718614718615 Full cohort mean: 8.028546712802768 They differ because the scaler sees only the randomly selected training rows.


## 6. Class-balance experiment (Task 7)
Duplicate minority-class rows **only in training**, refit logistic regression with its best C, and compare on the original test rows. More recall catches more lapsers, potentially at the cost of more unnecessary targeting.

In [11]:
balanced_logistic, balanced_metrics = balance_experiment(run)

Balanced train counts: {1: 570, 0: 570}
Original logistic: {'precision': 0.6444444444444445, 'recall': 0.651685393258427, 'f1': 0.6480446927374302}
Oversampled logistic: accuracy=0.716 precision=0.589 recall=0.854 F1=0.697
Confusion matrix [0,1]:
[[90 53]
 [13 76]]
Metric changes: {'precision': -0.055, 'recall': 0.202, 'f1': 0.049}


## 7. Interpret, save, reload and create retention list (Task 8)
Inspect forest importances and logistic coefficient signs; save the selected pipeline, verify identical predictions after reload, and rank Premium test customers. Default selects forest; choose `selected="logistic"` if that is better justified by your results. Premium is post-hoc and the resulting list is illustrative, **not** leakage-free evaluation.

In [12]:
retention, importances, coefficients = interpret_persist_retain(run, segments, selected='forest')
display(retention)

Top five forest features:
 num__frequency               0.279589
num__monetary                0.271176
num__recency                 0.115149
num__avg_delivery_minutes    0.061286
num__cancellation_rate       0.052325
Logistic coefficient signs (positive increases lapse odds):
 num__recency      0.039585
num__frequency   -1.698239
num__monetary     0.284573
Selected: forest test F1: 0.655367231638418
Reloaded predictions exactly match.
Premium test customers shortlisted: 10 truly lapsed: 1
LIMITATION: Premium is a post-hoc whole-window label including September; this retention list is illustrative, not a leakage-free evaluation.


,customer_id,lapse_probability,actually_lapsed
13,C0862,0.491035,0
27,C1112,0.484067,0
227,C0060,0.479796,0
5,C0802,0.479642,0
165,C1008,0.477817,0
121,C1000,0.462420,0
58,C0525,0.459718,1
82,C0133,0.456237,0
190,C0267,0.452514,0
16,C0167,0.452159,0


## Output checks
Verify that all three required files were created. Discuss the final model based on observed precision, recall, F1 and the retention team’s relative costs of missed lapsers versus wasted offers.

In [13]:
from pathlib import Path
for filename in ['customer_segments.csv','lapse_model.joblib','retention_list.csv']:
    path = ROOT /'outputs'/ filename
    print(path, 'exists:', path.exists())
    assert path.exists()

d:\Sowbarnika\Instamat_project\outputs\customer_segments.csv exists: True
d:\Sowbarnika\Instamat_project\outputs\lapse_model.joblib exists: True
d:\Sowbarnika\Instamat_project\outputs\retention_list.csv exists: True
